# FinSearch 2026 - Midterm Submission
## _Using Reinforcement Learning to Optimise Trading Strategies_
---
##### **Team:**
- ###### Ainesh Prakash
- ###### Sarvottam Dutt Chaturvedi
- ###### Shubham Mandal
- ###### Vipul Bansal


---
Deep Q-Network or DQN is one of the many reinforcement learning strategies used for basic systems and is quite beginner friendly to introduce someone to RL.


DQN is based on a classical concept called Q-learning in which the Q-function, $Q(s,a)$ calculates the expected total future reward if the agent is in state $s$, takes action $a$ and follows the optimal policy afterwards. In classical Q-learning the model tries to approximate $Q(s,a)$ for every possible $s$ and $a$ independently. It tries to build a sort of "lookup table" for every tuple $(s,a)$ which is directly stored in the model's memory.


Classical Q-learning gets too memory heavy as the environment on which we train the model gets more complex, it would be fine if you use Q-learning on a maze-solving model if the maze is tiled with squares in a 10x10 grid where the only actions are *left*, *right*, *up* or *down*. In this case the number of states are 100 with 4 actions on each state, making the domain of the Q-function a 4x100 matrix.


However if we try to use Q-learning on a game like Atari or Chess, the number of states and/or the number of action skyrockets, making classical Q-learning unfeasible.

DQN replaces the gigantic lookup table with a Deep Neural Network with the States as the input nodes and Actions as the output nodes and gives each Action a Q-value out of which we choose the action corresponding to the maximum Q-value which receives a reward $r$ from the environment.


The loss function $L(s,a,r)$ for the neural network is defined by comparing the current Q-value of action with the "ideal" Q-value the model thinks it should achieve.
The ideal Q-function is defined as,
$$
Q^*(s_t,a_t) = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \gamma^3 r_{t+3} + \dots
$$
where $\gamma$ is called the discount factor (between 0 and 1) which determines how much weight the model gives to future expected rewards

$$
\begin{align}
Q^*(s_t,a_t) &= r_t + \gamma (r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \dots) = r_t + \gamma Q^*(s_{t+1},a_{t+1}) \\
Q^*(s_t,a_t) &= r_t + \gamma \max_{a'} Q^*(s_{t+1},a')
\end{align}
$$

One of the core ideas in DQN is that the model can assume that $Q^*(s_{t+1},a') = Q(s_{t+1},a')$ since the model doesn't know any better. This is called Bootstrapping.

Hence we get the loss function,
$$
L(s,a,r) = \left[ \left( r + \gamma \max_{a'} Q(s^{++},a') \right) - Q(s,a) \right]^2
$$
where $s^{++}$ is the state reached after action $a$ on state $s$

To train the model using this loss function, we cannot simply feed the neural network every step exactly as it happens. If we do, the sequence of states will be highly correlated (since state $s^{++}$ is almost identical to state $s$), causing the gradient descent to become unstable.

Instead, DQN relies on an $\epsilon$-greedy policy to explore the environment and generate its own dataset. With a small probability $\epsilon$, the model picks a random action to explore. Otherwise, it exploits its current knowledge by passing $s$ through the neural network and choosing the action with the highest Q-value.

After taking the action, the model receives a reward $r$ and observes the next state $s^{++}$. Instead of calculating the loss immediately, the model takes this entire transition tuple $(s, a, r, s^{++})$ and stores it in a large memory array called Experience Replay.

Once the Experience Replay buffer has enough data, the training loop begins. The model samples a random mini-batch of transitions from the buffer. Because this batch is completely random, the sequential correlation is broken, stabilizing the gradient descent.

For every transition in this mini-batch, the model calculates the loss:

$$
L(s,a,r) = \left[ \left( r + \gamma \max_{a'} Q(s^{++},a') \right) - Q(s,a) \right]^2
$$

The model computes the gradients of this loss and uses backpropagation to update the neural network's weights. Over thousands of episodes, the $\epsilon$ value is slowly decayed, meaning the model gradually stops acting randomly and relies entirely on its trained neural network.

In [1]:
import torch
import torch.nn as nn

class DQN(nn.Module):
    def __init__(self, num_states, num_actions):
        super().__init__()

        self.layers = torch.nn.Sequential(
            # 1st hidden layer
            nn.Linear(num_states, 128),
            nn.ReLU(),

            # 2nd hidden layer
            nn.Linear(128, 32),
            nn.ReLU(),

            # output layer
            nn.Linear(32, num_actions)
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [2]:
import random
from collections import deque

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)

In [3]:
import torch.optim as optim

class DQNAgent:
    def __init__(self, num_states, num_actions):
        self.num_actions = num_actions
        self.num_states = num_states
        self.batch_size = 64
        self.gamma = 0.99

        self.network = DQN(num_states, num_actions)
        self.memory = ReplayBuffer(capacity=10000)
        self.optimizer = optim.Adam(self.network.parameters(), lr=0.001)

    def act(self, state, epsilon):
        if random.random() < epsilon:
            return random.randint(0, self.num_actions - 1)

        state_tensor = torch.tensor(state, dtype=torch.float32)
        with torch.no_grad():
            q_values = self.network(state_tensor)
            return torch.argmax(q_values).item()

    def train(self):
        if len(self.memory) < self.batch_size:
            return

        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)

        states = torch.tensor(np.array(states), dtype=torch.float32)
        actions = torch.tensor(actions, dtype=torch.int64).unsqueeze(1)
        rewards = torch.tensor(rewards, dtype=torch.float32).unsqueeze(1)
        next_states = torch.tensor(np.array(next_states), dtype=torch.float32)
        dones = torch.tensor(dones, dtype=torch.float32).unsqueeze(1)

        current_q_values = self.network(states).gather(1, actions)
        with torch.no_grad():
            max_next_q_values = self.network(next_states).max(1)[0].unsqueeze(1)

        target_q_values = rewards + (self.gamma * max_next_q_values * (1 - dones))
        loss = nn.functional.mse_loss(current_q_values, target_q_values)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

In [4]:
import gymnasium as gym
import numpy as np

env = gym.make('Pendulum-v1')
discrete_actions = np.linspace(-2.0, 2.0, 11)
agent = DQNAgent(num_states=3, num_actions=11)

epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.05

for episode in range(1500):
    state, info = env.reset()
    total_reward = 0
    done = False

    while not done:
        action_index = agent.act(state, epsilon)
        torque = discrete_actions[action_index]

        next_state, reward, terminated, truncated, info = env.step([torque])
        done = terminated or truncated
        total_reward += reward

        agent.memory.push(state, action_index, reward, next_state, done)
        agent.train()

        state = next_state

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    print(f"Episode {episode} | Reward: {total_reward:.2f}")

Episode 0 | Reward: -1058.95
Episode 1 | Reward: -1173.78
Episode 2 | Reward: -1420.98
Episode 3 | Reward: -967.83
Episode 4 | Reward: -977.34
Episode 5 | Reward: -1624.20
Episode 6 | Reward: -1833.32
Episode 7 | Reward: -900.36
Episode 8 | Reward: -945.70
Episode 9 | Reward: -1162.75
Episode 10 | Reward: -1301.53
Episode 11 | Reward: -897.56
Episode 12 | Reward: -871.80
Episode 13 | Reward: -1150.66
Episode 14 | Reward: -856.60
Episode 15 | Reward: -965.16
Episode 16 | Reward: -1120.08
Episode 17 | Reward: -897.79
Episode 18 | Reward: -1416.91
Episode 19 | Reward: -998.41
Episode 20 | Reward: -778.30
Episode 21 | Reward: -967.13
Episode 22 | Reward: -1297.81
Episode 23 | Reward: -1397.48
Episode 24 | Reward: -1074.62
Episode 25 | Reward: -1083.21
Episode 26 | Reward: -1096.17
Episode 27 | Reward: -870.46
Episode 28 | Reward: -884.23
Episode 29 | Reward: -1002.71
Episode 30 | Reward: -1387.27
Episode 31 | Reward: -1284.24
Episode 32 | Reward: -1341.72
Episode 33 | Reward: -1560.79
Epis

In [ ]:
torch.save(agent.network.state_dict(), './DQN.pth')

In [7]:
env = gym.make("Pendulum-v1", render_mode="human")

for episode in range(5):
    state, info = env.reset()
    done = False
    total_reward=0

    while not done:
        action_index = agent.act(state, epsilon=0)
        torque = discrete_actions[action_index]

        state, reward, terminated, truncated, info = env.step([torque])
        done = terminated or truncated
        total_reward += reward
    print(f"Episode {episode+1} | Reward: {total_reward:.2f}")
env.close()

Episode 1 | Reward: -0.42
Episode 2 | Reward: -242.66
Episode 3 | Reward: -124.49
Episode 4 | Reward: -114.89
Episode 5 | Reward: -118.77
